# Assignment 19: Word2Vec Text Embeddings
**Student:** Parth Dadhaniya  
**Course:** GenAI Course  

### Problem Statement
Traditional vectorization techniques (BoW, TF-IDF) fail to capture semantic meaning and context between words. In this assignment, we implement Word2Vec, explore CBOW and Skip-gram architectures, understand the neural network weight intuition, and train models on custom cleaned text.

## PART 1 — Introduction to Word Embeddings
### Task 1: Understanding Word Embeddings (Conceptual)

In [ ]:
# Task 1: Conceptual understanding
print('1. What are word embeddings?')
print('- Word embeddings are dense numerical vectors that represent words in a continuous space.')
print('- Words with similar meanings end up close to each other in this vector space.')
print('- Unlike one-hot vectors which are huge and sparse, embeddings use small dense dimensions like 50, 100, or 300.\n')

print('2. Why One-Hot Encoding and BoW fail to capture semantics:')
print('- They treat every word as completely independent and perpendicular to other words.')
print('- The dot product between any two distinct one-hot vectors is always 0, so king and queen have zero similarity.')
print('- They ignore word order, context, and meaning.')
print('- Vocabulary size makes vectors huge and mostly filled with zeros.\n')

print('3. How word embeddings solve these problems:')
print('- They use low-dimensional dense vectors (e.g. 100 numbers instead of 10,000 zeros).')
print('- Words used in similar contexts share similar vector coordinates, so cosine similarity is high.')
print('- They capture relationships with vector math, like: king - man + woman = queen.')
print('- Models generalize much better to new sentences having synonyms.')

## PART 2 — Word2Vec Overview & Architectures
### Task 2: Word2Vec Overview

In [ ]:
# Task 2: Word2Vec Overview
print('1. What is Word2Vec?')
print('- Word2Vec is a shallow neural network model introduced by Tomas Mikolov and Google in 2013.')
print('- It learns word vectors by scanning text and observing which words appear next to each other.\n')

print('2. The Core Idea: Predicting Words from Context')
print('- It follows the idea: You shall know a word by the company it keeps.')
print('- Word2Vec slides a small window across text and trains a network to predict target words from context words.\n')

print('3. Key Definitions:')
print('a) Vocabulary (V): All unique words in the text dataset meeting minimum count threshold.')
print('b) Context Window: Number of words before and after target word to consider as context.')
print('c) Embedding Dimension (d): Size of the dense vector for each word (like 50, 100, or 300).')

### Task 3: Types of Word2Vec Techniques (CBOW vs Skip-Gram)

In [ ]:
# Task 3: CBOW vs Skip-Gram comparison
print('1. Continuous Bag of Words (CBOW):')
print('- How it works: CBOW takes surrounding context words and predicts the center word.')
print('- Example: Context inputs [queen, the] -> Model -> Output target: ruled')
print('- Key points: Averages context vectors. Fast training, works well for frequent words.\n')

print('2. Skip-Gram:')
print('- How it works: Skip-Gram takes a center word and predicts each surrounding context word.')
print('- Example: Center input ruled -> Model -> Context outputs [queen, the]')
print('- Key points: Creates multiple pairs. Slower training, but does much better on rare words.\n')

print('3. When to use each:')
print('- Use CBOW if you have huge datasets and want fast training.')
print('- Use Skip-Gram if you have small/medium datasets or care about rare words.')

### Task 4: Neural Network Intuition Behind Word2Vec

In [ ]:
# Task 4: Neural Network Intuition
print('1. Architecture: Input Layer (1xV) -> Hidden Layer (1xd) -> Output Layer (1xV)')
print('   Input Layer: 1-hot vector selects row i of weight matrix W (size V x d).')
print('   Hidden Layer: Linear projection with no activation function.')
print('   Output Layer: Softmax converts logits into probabilities over vocabulary.')
print('2. How weights become embeddings: Matrix W is the lookup table; row k is vector for word k.')

## PART 3 — Training Word2Vec on Custom Data
### Task 5: Prepare Text for Word2Vec

In [ ]:
# Task 5: Load and tokenize text
import pandas as pd

df = pd.read_csv('cleaned_text_corpus.csv')
sentences = [str(t).strip().split() for t in df['clean_text'] if str(t).strip()]

print('Total sentences:', len(sentences))
print('Sample tokenized sentence 1:', sentences[0])

### Task 6: Train CBOW Word2Vec Model (`sg=0`)

In [ ]:
# Task 6: Train CBOW Word2Vec
from gensim.models import Word2Vec

model_cbow = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    sg=0,
    seed=42,
    epochs=50
)

print('CBOW Vocabulary Size:', len(model_cbow.wv.key_to_index))
print('Vector for movie (first 10 values):\n', model_cbow.wv['movie'][:10])

### Task 7: Skip-Gram Word2Vec Model (`sg=1`) & Comparison

In [ ]:
# Task 7: Train Skip-Gram and compare with CBOW
import time

t0 = time.time()
model_cbow = Word2Vec(sentences, vector_size=100, window=5, min_count=1, sg=0, seed=42, epochs=50)
t_cbow = time.time() - t0

t1 = time.time()
model_sg = Word2Vec(sentences, vector_size=100, window=5, min_count=1, sg=1, seed=42, epochs=50)
t_sg = time.time() - t1

print(f'CBOW Training Time:      {t_cbow * 1000:.2f} ms')
print(f'Skip-Gram Training Time: {t_sg * 1000:.2f} ms\n')

for w in ['movie', 'king', 'phone']:
    print(f'Word: {w}')
    print('  CBOW:     ', [sim_w for sim_w, _ in model_cbow.wv.most_similar(w, topn=3)])
    print('  Skip-Gram:', [sim_w for sim_w, _ in model_sg.wv.most_similar(w, topn=3)])

### Task 8: Word Similarity & Vector Operations

In [ ]:
# Task 8: Similarity and Vector Operations
wv = model_sg.wv

print('1. Most similar to king:', wv.most_similar('king', topn=3))

print('\n2. Vector arithmetic (king - man + woman):')
analogy = wv.most_similar(positive=['king', 'woman'], negative=['man'], topn=3)
for w, score in analogy:
    print(f'  {w}: {score:.4f}')

print('\n3. Pairwise Cosine Similarity:')
print('  movie <-> film:', round(wv.similarity('movie', 'film'), 4))
print('  king <-> queen:', round(wv.similarity('king', 'queen'), 4))
print('  king <-> battery (unrelated):', round(wv.similarity('king', 'battery'), 4))

print('\n4. Odd-One-Out Detection:')
print('  [phone, laptop, battery, king] -> Outlier:', wv.doesnt_match(['phone', 'laptop', 'battery', 'king']))

## PART 4 — Evaluation & Insights
### Task 9: Visualizing Word Embeddings (PCA 2D)

In [ ]:
# Task 9: 2D PCA Visualization
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

groups = {
    'Cinema': ['movie', 'film', 'actor', 'actress', 'cinema', 'director'],
    'Tech': ['phone', 'battery', 'laptop', 'screen', 'keyboard', 'device'],
    'Support': ['customer', 'service', 'support', 'refund', 'ticket', 'order'],
    'Royalty': ['king', 'queen', 'prince', 'princess', 'palace', 'kingdom']
}

words, colors = [], []
cmap = {'Cinema': 'red', 'Tech': 'blue', 'Support': 'green', 'Royalty': 'purple'}

for cat, w_list in groups.items():
    for w in w_list:
        if w in wv:
            words.append(w)
            colors.append(cmap[cat])

coords = PCA(n_components=2, random_state=42).fit_transform([wv[w] for w in words])

plt.figure(figsize=(9, 6))
for i, w in enumerate(words):
    plt.scatter(coords[i, 0], coords[i, 1], color=colors[i], s=70)
    plt.annotate(w, (coords[i, 0], coords[i, 1]), textcoords='offset points', xytext=(5, 5))

for cat, color in cmap.items():
    plt.scatter([], [], color=color, label=cat)
plt.legend(title='Category')
plt.title('2D Projection of Word Embeddings (PCA)')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

### Task 10: Technical Observations & Limitations

In [ ]:
# Task 10: Observations & Limitations
print('1. CBOW vs Skip-Gram in practice: CBOW trains faster and averages context; Skip-Gram handles rare words better.')
print('2. Word2Vec vs TF-IDF: Word2Vec provides dense semantic vectors and vector arithmetic, unlike TF-IDF orthogonal sparse vectors.')
print('3. Limitations: Fixed static embeddings (cannot handle polysemy like apple fruit vs company) and OOV issues.')
print('4. Modern NLP Context: Transformers (BERT/GPT) solve static limitations by computing dynamic contextual embeddings via self-attention.')